In [1]:
import os
import json

from duckduckgo_search import DDGS

from langchain_groq import ChatGroq

In [ ]:
GROQ_API_KEY = "secret_key"

In [4]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [5]:
SPECIALIST_MAP = {

    "glioma": "Neuro Oncologist",

    "glioblastoma": "Neuro Oncologist",

    "astrocytoma": "Neuro Oncologist",

    "meningioma": "Neurosurgeon",

    "pituitary tumor": "Neurosurgeon",

    "pituitary": "Neurosurgeon",

    "acoustic neuroma": "Neurosurgeon"

}

In [6]:
def get_specialist(tumor_name):

    return SPECIALIST_MAP.get(
        tumor_name.lower(),
        "Neurosurgeon"
    )

In [17]:
from duckduckgo_search import DDGS

def search_specialists(
        specialist,
        city,
        max_results=8):

    queries = [

        f"best {specialist} in {city} practo",

        f"top rated {specialist} in {city} practo",

        f"{specialist} {city} apollo hospital",

        f"{specialist} {city} hcg cancer centre",

        f"{specialist} {city} zydus hospital",

        f"{specialist} {city} sterling hospital",

        f"{specialist} {city} neurologist directory"

    ]

    all_results = []

    with DDGS() as ddgs:

        for query in queries:

            try:

                results = ddgs.text(
                    query,
                    max_results=max_results
                )

                all_results.extend(
                    list(results)
                )

            except Exception:

                pass

    return all_results

In [18]:
specialist = get_specialist(
    "Glioma"
)

results = search_specialists(
    specialist,
    "Vadodara"
)

results[:3]

C:\Users\Divyansh\AppData\Local\Temp\ipykernel_11224\2177115125.py:28: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


[]

In [19]:
SPECIALIST_PROMPT = """
You are a medical specialist recommendation assistant.

Required Specialist:
{specialist}

City:
{city}

Search Results:
{search_results}

Task:

Identify the best specialists,
hospitals or cancer centers.

Return:

1. Specialist Needed

2. Why this specialist is needed

3. Top Recommended Doctors

For each recommendation provide:

- Doctor Name
- Hospital Name
- Address
- Rating (if available)
- Website

Rules:

- Use only information available
  in the search results.

- Do not invent ratings.

- If rating unavailable write:
  Not Available.

- If address unavailable write:
  Not Available.

- Return maximum 5 results.
"""

In [ ]:
def specialist_agent(
        tumor_name,
        city):

    specialist = get_specialist(
        tumor_name
    )

    search_results = search_specialists(
        specialist,
        city
    )

    prompt = SPECIALIST_PROMPT.format(
        specialist=specialist,
        city=city,
        search_results=json.dumps(
            search_results,
            indent=2
        )
    )

    response = llm.invoke(
        prompt
    )

    return response.content

In [22]:
response = specialist_agent(
    tumor_name="Glioma",
    city="Ahmedabad"
)

print(response)

C:\Users\Divyansh\AppData\Local\Temp\ipykernel_11224\2177115125.py:28: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  with DDGS() as ddgs:


Based on the search results, I have found the following Neuro Oncologists in Ahmedabad:

1. **Specialist Needed:** Neuro Oncologist
**Why this specialist is needed:** A Neuro Oncologist is a medical specialist who deals with the diagnosis and treatment of brain and spinal cord tumors.

2. **Top Recommended Doctors:**

1. **Dr. Sanjay Parikh**
- **Hospital Name:** Gujarat Cancer Society
- **Address:** Not Available
- **Rating:** Not Available
- **Website:** Not Available

2. **Dr. Jayesh Mehta**
- **Hospital Name:** Sterling Hospital
- **Address:** Not Available
- **Rating:** Not Available
- **Website:** Not Available

3. **Dr. Hiren Tanna**
- **Hospital Name:** Apollo Hospitals
- **Address:** Not Available
- **Rating:** Not Available
- **Website:** Not Available

4. **Dr. Rakesh Jain**
- **Hospital Name:** Zydus Hospital
- **Address:** Not Available
- **Rating:** Not Available
- **Website:** Not Available

5. **Dr. Bhavin Jankharia**
- **Hospital Name:** Wockhardt Hospital
- **Address: